<a href="https://colab.research.google.com/github/niranjanniru-max/IRIS-Classification-Dataset/blob/main/IRIS_Classification_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[hf-datasets] pandas
import kagglehub
import pandas as pd
import os

# The name of the specific file within the dataset you want to load
file_name = "Iris.csv"  # Note: The file name in 'uciml/iris' is typically 'Iris.csv' with a capital 'I'

# Download the dataset. This caches the dataset locally and returns the path to its directory.
dataset_dir = kagglehub.dataset_download("uciml/iris")

# Construct the full path to the specific file
file_path = os.path.join(dataset_dir, file_name)

# Load the dataset into a pandas DataFrame
data = pd.read_csv(file_path)
print("Dataset loaded into DataFrame (first 5 rows):")

Using Colab cache for faster access to the 'iris' dataset.
Dataset loaded into DataFrame (first 5 rows):


In [ ]:
print("Dataset loaded into DataFrame (first 5 rows):")
print(data.head())
print(data.describe())
print(data.info())

Dataset loaded into DataFrame (first 5 rows):
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa
               Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm
count  150.000000     150.000000    150.000000     150.000000    150.000000
mean    75.500000       5.843333      3.054000       3.758667      1.198667
std     43.445368       0.828066      0.433594       1.764420      0.763161
min      1.000000       4.300000      2.000000       1.000000      0.100000
25%     38.250000       5.100000      2.800000       1.600000      0.300000
50%     75.500000       5.800000    

In [ ]:
# Drop identifier and target columns
X = data.drop(['Id', 'Species'], axis=1)
y = data['Species']

# Train/test split
from sklearn.model_selection import train_test_split, GridSearchCV
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

In [ ]:

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
num = X.select_dtypes(include=['int64','float64']).columns
cat = X.select_dtypes(include=['object']).columns
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num),
    ('cat', OneHotEncoder(), cat)
])
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

param_grid = [
    {
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': [0.1, 1, 10],
        'model__solver': ['liblinear', 'lbfgs']
    },
    {
        'model': [RandomForestClassifier()],
        'model__n_estimators': [100, 200],
        'model__max_depth': [10, 20, None],
        'model__min_samples_split': [2, 5],
        'model__max_features': ['sqrt', 'log2']
    },
    {
        'model': [DecisionTreeClassifier()],
        'model__max_depth': [10, 20, None],
        'model__min_samples_split': [2, 5],
        'model__criterion': ['gini', 'entropy']
    }
]

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

y_pred = grid_search.predict(X_test)



In [ ]:

# Results
best_pipeline = grid_search.best_estimator_
best_model = best_pipeline.named_steps['model']
test_score = best_pipeline.score(X_test, y_test)

print("Best Model:", best_model)
print("Best params:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)
print("Test Accuracy:", test_score)

print(classification_report(y_test, y_pred))

print("Features used:", X_train.columns)

Best Model: LogisticRegression(C=10, max_iter=1000)
Best params: {'model': LogisticRegression(max_iter=1000), 'model__C': 10, 'model__solver': 'lbfgs'}
Best CV Accuracy: 0.9666666666666668
Test Accuracy: 1.0
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      1.00      1.00        10
 Iris-virginica       1.00      1.00      1.00        10

       accuracy                           1.00        30
      macro avg       1.00      1.00      1.00        30
   weighted avg       1.00      1.00      1.00        30

Features used: Index(['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'], dtype='object')


✨ I implemented the Iris classification task myself, and my version goes beyond the one in the PDF. Their version mainly used LabelEncoder/MinMaxScaler, separate steps, and focused on Logistic Regression with lots of visualizations. Mine is different: I built a proper ML pipeline with ColumnTransformer (StandardScaler + OneHotEncoder), compared three models (Logistic Regression, Random Forest, Decision Tree), and tuned hyperparameters using GridSearchCV. I reported CV score, best params, classification report, and test accuracy — so it’s structured, reusable, and closer to how ML is done in production.

The major difference is that their version is more exploratory with plots, while mine is engineering‑oriented with pipelines and metrics. I focused on numbers because that’s how I understand results best, even though I’m still improving at visualization. I’ve studied ML thoroughly during this internship, and this project reflects that learning.